# Instructional Notebook for SHRED + Biomechanics

The following lines can be uncommented if running this notebook in Google Colab. Uncomment by highlighting lines and pressing Ctrl+/

In [49]:
#from google.colab import drive
#drive.mount('/content/drive')
#!pip install mat73
#!git clone https://github.com/Jan-Williams/pyshred
#%cd /content/pyshred

These lines import standard packages for managing data.

In [50]:
import os
import numpy as np
import altair as alt
import pandas as pd
from processdata import TimeSeriesDataset
import models
import torch
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
import mat73
import functions as ft

# *** Update ***
Load a subject's data and manipulate the dataframe to be "tidy" = one row per time step and one column per signal. Depending on the dataset, loading and managing data will look different.

## Obtain subject's experimental data

Items to adjust before running a trial:

*   save_df - True (save) or False (don't save)
*   subject - '##'
*   activity code - AC## (this may need a different identifier depending on the dataset, just need a way to distinguish running speeds)

In [51]:
# Change subject number
subj = '03'    # 01-09
save_df = False   # True: save SHRED output dataframes only, False: don't save
trial_length = 5  # in minutes, accepts integers 1-6
frequency = 128 # in Hz, accepts integers up to 128

# adjust file path for saving if parameters are modified from 6min or 128Hz
if trial_length == 6:
    save_tag = str(frequency)+'Hz'
elif frequency == 128:
    save_tag = str(trial_length)+'min'


## Import Matlab file structure with subject's experimental data

Access directories where data is stored and will be saved. Manually set up folders before running the code block to ensure known file paths.

In [52]:
cwd = os.getcwd()
main_path = os.path.dirname(cwd) + '/Datasets' 

# Alternatively, use the below lines if using Colab
# main_path = '/content/drive/MyDrive/Colab_Notebooks/Datasets'
# main_path = cwd+'/Datasets'

dataset_path = main_path+'/Data' # sets path to dataset / raw data
dataframe_path = main_path+'/Dataframes'  # file path for saved dataframe results of test data
figure_path = main_path+'/Figures'
model_path = main_path+'/Models' # optionally, save the models that are trained
print(dataset_path)
print(dataframe_path)


/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Data
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes


Note on changing file directory and needing to save the parent directory:

https://stackoverflow.com/questions/14462833/how-can-i-go-back-to-the-previous-working-directory-after-changing-it

In [53]:
# load .mat file into pandas dataframe
load_mat = mat73.loadmat(dataset_path+'/Subject'+subj+'.mat')['Subject'+subj]
df = pd.DataFrame.from_dict(load_mat)

The example dataset contains several activities, two of which are 'Walking' and 'Running'. Access each independently.

In [54]:
#df_tmp = pd.DataFrame(data=df['Walking']['APDM_Accel']['Data'],
#                      columns = df['Walking']['APDM_Accel']['Labels'])

df_tmp = pd.DataFrame(data=df['Running']['APDM_Accel']['Data'],
                      columns = df['Running']['APDM_Accel']['Labels'])

pd.set_option('display.max_columns', None)

df_tmp.tail(5) # check that correct data was selected

Time (s) Activity Code                Waist                      \
                                  Acceleration (m/s^2)                       
                                                     x         y         z   
276643  2161.196828          23.0            -8.721255 -1.259308  4.721980   
276644  2161.204640          23.0            -8.554564 -1.292766  4.565858   
276645  2161.212453          23.0            -8.651367 -1.231922  4.449909   
276646  2161.220265          23.0            -8.689855 -1.251493  4.656814   
276647  2161.228077          23.0            -8.624153 -1.225631  4.338572   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276643                 0.065694 -0.082697 -0.024596           44.069507   
276644                 0.053182 -0.040165 -0.017693           44.197406   
276645                 0.070335 -0.069796 -0.018018           44.427027   
276646                 0.048493 -0.045698 -0.017513           44.381427   
276647                 0.065621 -0.043642 -0.011636           44.393715   

                                           Chest                      \
                            Acceleration (m/s^2)                       
                y         z                    x         y         z   
276643  19.748847 -3.982887            -7.715366  2.584175  5.551181   
276644  19.778696 -4.010292            -7.743191  2.597559  5.555703   
276645  19.628141 -4.004308            -7.743300  2.595075  5.555658   
276646  19.795449 -4.001773            -7.763569  2.586867  5.537731   
276647  19.671439 -4.022836            -7.768969  2.587482  5.505917   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276643                 0.106575  0.077133 -0.013045           51.959348   
276644                 0.105137  0.077149 -0.006612           51.785684   
276645                 0.106832  0.074140 -0.004961           51.794751   
276646                 0.105154  0.074070 -0.006624           51.918650   
276647                 0.107464  0.076015 -0.007812           51.888234   

                                       Left Ankle                      \
                             Acceleration (m/s^2)                       
                y          z                    x         y         z   
276643  34.164922 -25.397757            -8.820062 -4.828279  0.145405   
276644  34.138160 -25.559350            -8.765298 -4.981721  0.325385   
276645  34.147235 -25.554256            -8.730404 -5.042830  0.343587   
276646  34.121663 -25.420862            -8.793706 -5.070813  0.168257   
276647  34.251927 -25.426268            -8.658033 -5.088742  0.350552   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276643                 0.058112  0.033767  0.065080           30.209212   
276644                 0.098407  0.016559  0.059262           30.393037   
276645                 0.136695  0.001245  0.055024           30.254478   
276646                 0.199051 -0.011608  0.038394           30.150357   
276647                 0.275878 -0.039115  0.013935           30.177220   

                                      Right Ankle                      \
                             Acceleration (m/s^2)                       
                y          z                    x         y         z   
276643  14.520742  10.514054            -7.938647  5.574151  0.076650   
276644  14.509587  10.783967            -7.932501  5.463531  0.161563   
276645  14.514723  10.891705            -7.951330  5.504

In [55]:
# remove units and simplify column titles
columns_str = ["_".join(df_tmp.columns[i]).replace(" ", "") for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = columns_str

l_replace = [df_tmp.columns[i].replace('(m/s^2)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(rad/s)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(uT)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace
df_tmp.head()

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,LeftFoot_Acceleration_x,LeftFoot_Acceleration_y,LeftFoot_Acceleration_z,LeftFoot_AngularVelocity_x,LeftFoot_AngularVelocity_y,LeftFoot_AngularVelocity_z,LeftFoot_MagneticField_x,LeftFoot_MagneticField_y,LeftFoot_MagneticField_z,RightFoot_Acceleration_x,RightFoot_Acceleration_y,RightFoot_Acceleration_z,RightFoot_AngularVelocity_x,RightFoot_AngularVelocity_y,RightFoot_AngularVelocity_z,RightFoot_MagneticField_x,RightFoot_MagneticField_y,RightFoot_MagneticField_z
0,0.007812,22.0,-8.962763,-1.100030,3.936183,0.087325,-0.052631,0.004653,34.081727,16.055235,-7.675515,-9.162053,0.070575,3.718113,0.088554,0.112437,0.000549,59.294744,4.023130,-1.820654,-9.505575,-1.949894,-0.999578,0.139062,0.027139,0.024772,31.691974,-1.674657,16.771539,-9.735184,1.085768,-0.674665,-0.183196,-0.160582,-0.018959,32.816336,-3.309245,-12.546763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.015624,22.0,-8.931067,-1.218816,3.956412,0.098277,-0.067036,-0.000214,33.995754,15.944795,-8.082310,-9.150810,0.058742,3.713305,0.098301,0.103577,0.005381,59.118798,4.017682,-1.678183,-9.503029,-1.949952,-1.004229,0.149149,0.028507,0.026541,31.593215,-1.647272,16.708203,-9.725285,1.074768,-0.651423,-0.175210,-0.165708,-0.015978,32.715914,-3.116667,-12.630243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.023437,22.0,-8.929444,-1.137145,3.914310,0.098262,-0.062162,0.002562,34.136300,15.968446,-7.914372,-9.139326,0.054146,3.724545,0.102960,0.106740,-0.002549,59.130781,4.032866,-1.551405,-9.525234,-1.942909,-0.986118,0.140400,0.028094,0.019995,31.488555,-1.792291,16.714591,-9.739121,1.079349,-0.674490,-0.178346,-0.160819,-0.022437,32.589109,-3.244219,-12.878493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.031249,22.0,-8.984057,-1.156247,3.967663,0.099849,-0.074139,-0.002094,34.144614,15.967275,-7.701415,-9.143631,0.075008,3.720459,0.103951,0.103173,0.002115,59.244058,3.964650,-1.655923,-9.528324,-1.936088,-0.983613,0.146845,0.030286,0.023311,31.483909,-1.687465,16.745081,-9.740721,1.082032,-0.676519,-0.181266,-0.161205,-0.024324,32.683737,-3.370395,-12.858772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.039061,22.0,-8.919969,-1.201421,3.916294,0.109170,-0.059234,0.004619,33.988565,15.971928,-7.610035,-9.140747,0.060156,3.734255,0.109846,0.089003,0.011557,59.271157,3.722052,-1.925677,-9.525851,-1.940618,-0.992779,0.151144,0.030196,0.021780,31.618774,-1.470487,16.581184,-9.739351,1.079217,-0.674581,-0.178418,-0.154321,-0.024056,32.952758,-3.200268,-12.705910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Obtain data with desired activity code

In [56]:
# subject running codes [12, 13, 14] = 1.8, 2,2, 2.7 m/s
AC_path = 'AC14'
df_2=df_tmp.loc[df_tmp['ActivityCode__']==13].dropna(axis=1,how='all')
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
92163,720.003125,13.0,-7.968292,-1.536261,1.813941,0.896101,0.063596,-0.183284,32.420663,17.211843,-4.605510,-8.566226,1.287806,2.238931,1.580963,0.634100,-0.238720,58.809461,4.858203,-0.310155,-9.498444,-1.643821,-2.424220,0.236637,-0.082041,0.567447,24.021110,-2.763524,21.154090,-4.382144,3.436032,0.053419,-0.557730,-0.058931,3.360149,11.956045,11.998172,-14.102421
92164,720.010937,13.0,-7.928762,-1.457954,1.775878,0.960206,0.122740,-0.186564,32.646821,17.426894,-4.982493,-8.244154,1.402774,2.395651,1.559618,0.672972,-0.303709,59.012394,4.740188,0.001505,-9.468582,-1.539004,-2.351286,0.326210,-0.047509,0.602733,23.970284,-2.641518,21.018676,-4.252664,2.749348,0.328891,-0.666331,-0.090359,3.680345,12.449225,12.054163,-13.999114
92165,720.018749,13.0,-7.686839,-1.299204,1.711628,1.008582,0.221780,-0.185064,32.486827,17.348844,-5.026693,-7.837158,1.600472,2.451163,1.533776,0.749642,-0.348384,59.290582,4.766603,0.042781,-9.641674,-1.608247,-2.512936,0.387406,-0.034663,0.631237,23.764140,-2.538335,21.022636,-4.492818,2.090735,0.418436,-0.697160,-0.119697,4.011293,13.047044,12.038704,-13.916088
92166,720.026561,13.0,-7.517695,-1.185442,1.502237,1.044446,0.259074,-0.178111,32.439894,17.529719,-5.099916,-7.487098,1.730786,2.348739,1.406393,0.821192,-0.381748,59.502581,4.768507,0.221568,-9.437702,-1.280018,-2.599629,0.400382,-0.032968,0.629942,23.588334,-2.578671,21.139683,-4.955268,2.046597,0.553444,-0.676845,-0.172143,4.334743,13.924944,11.435313,-13.754394
92167,720.034374,13.0,-7.259002,-1.213567,1.796864,1.069274,0.269988,-0.162335,32.202348,17.849781,-5.369755,-7.226017,1.755910,2.194974,1.230376,0.839308,-0.348847,59.359554,4.752795,0.642073,-9.185970,-0.405883,-2.273790,0.407148,0.016540,0.633503,23.412405,-2.461162,21.207834,-5.584008,2.595451,0.607606,-0.644204,-0.253991,4.609085,14.473554,10.897219,-13.262630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138240,1079.965626,13.0,-18.229356,0.024519,2.417356,-2.998924,-0.890793,0.371328,38.416287,10.314729,1.184247,-5.881049,-3.868557,11.655907,-1.176077,-1.188272,2.702903,58.044540,26.080097,5.584343,-5.484031,-5.405044,-5.335284,0.794465,-2.011347,-9.611934,1.082726,-25.105176,16.739606,-30.048477,4.258544,-2.326319,-0.821886,-1.550859,-2.450101,17.881782,7.909293,-6.424720
138241,1079.973439,13.0,-16.347993,0.356063,1.726132,-3.019206,-0.874072,0.369610,38.630940,10.208115,1.251905,-6.509489,-3.226304,9.915414,-0.952814,-0.255495,2.837007,58.352202,25.348942,5.154383,-9.862419,-5.322267,-4.816580,0.679191,-1.928497,-9.800714,3.794034,-25.074653,16.804234,-32.180863,7.087209,-2.269239,-0.486708,-1.732076,-2.223136,17.446077,7.989878,-6.761471
138242,1079.981251,13.0,-14.399766,0.916423,1.128389,-3.110302,-0.859553,0.248078,38.853007,10.105693,1.782533,-6.674558,-2.471423,6.993199,-0.803550,0.553607,2.883820,58.829264,24.523080,5.223109,-13.130319,-4.396824,-4.824985,0.623346,-1.756815,-10.142833,6.100014,-24.655264,16.739531,-30.130298,5.811476,-2.732883,0.178730,-1.44452

### Simplify dataframe

In [57]:
# trim length of trial (number of rows in df)
obs_samples_trial = trial_length*60*frequency
df_2 = df_2.tail(obs_samples_trial) # keep last n samples to exclude speed transitions

# downsample trial
obs_samples_freq = int(128/frequency)

df_2 = df_2.iloc[::obs_samples_freq,:]
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
99845,780.016406,13.0,-21.723717,-0.906709,-3.340406,-1.725156,1.383336,0.336675,37.023064,12.937910,-3.726508,-13.029014,2.780933,8.791527,-0.382982,-1.343675,-0.590641,59.231463,9.135267,10.539824,-7.421039,-11.276883,-7.374559,1.016987,-0.353633,1.914961,1.570668,-24.163764,12.631219,-5.127416,-3.676296,-3.275163,0.393977,-0.165850,-2.720616,23.882844,4.893803,-1.655445
99846,780.024218,13.0,-26.138784,0.536051,2.104222,-2.643493,0.152196,0.345214,37.042200,13.303920,-3.233381,-19.080092,-4.158114,5.273474,-0.303717,0.264697,-0.726026,59.229017,9.047679,10.011694,-5.934692,-6.664719,-8.358840,1.667743,-0.268055,1.409116,0.899844,-24.119268,12.564553,-2.301847,-4.055829,-6.174088,0.606819,-0.320822,-2.267808,23.178881,5.153526,-1.746373
99847,780.032030,13.0,-30.132945,3.394068,0.663351,-2.598406,0.614315,0.611814,36.813951,13.891790,-2.678930,-26.234258,-11.518528,2.172397,0.175134,0.943130,-0.927280,59.133957,9.197701,9.930569,-4.640564,-1.135332,-9.015020,2.075752,-0.265863,0.791366,0.318406,-24.269342,12.862920,-6.659033,-6.532508,-2.847930,0.738536,-0.389486,-1.764837,22.323508,5.576772,-1.843484
99848,780.039842,13.0,-34.909282,2.884061,-1.588547,-2.502476,0.360988,0.486871,36.807108,14.106254,-1.983703,-29.278941,-1.742820,9.607354,1.012114,1.254744,-1.325094,58.996962,9.665787,10.196931,-3.737068,2.567238,-7.985579,2.310795,-0.278481,0.021561,0.128372,-24.128601,13.414327,-12.390128,-7.936329,-2.243952,0.631756,-0.313315,-1.445026,21.750232,5.577661,-1.967108
99849,780.047654,13.0,-33.515009,2.409845,1.141210,-2.055668,0.487387,0.452026,37.177622,14.261348,-1.393292,-41.144383,-0.896207,10.917825,0.764393,3.136922,-0.738035,59.157526,9.984614,10.279470,-3.277699,3.577035,-6.633867,2.444098,-0.345753,-0.809418,-0.228798,-24.040231,13.630038,-11.429309,-6.475535,-4.453832,0.683026,-0.443690,-1.215093,21.108702,5.767338,-1.924253
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138240,1079.965626,13.0,-18.229356,0.024519,2.417356,-2.998924,-0.890793,0.371328,38.416287,10.314729,1.184247,-5.881049,-3.868557,11.655907,-1.176077,-1.188272,2.702903,58.044540,26.080097,5.584343,-5.484031,-5.405044,-5.335284,0.794465,-2.011347,-9.611934,1.082726,-25.105176,16.739606,-30.048477,4.258544,-2.326319,-0.821886,-1.550859,-2.450101,17.881782,7.909293,-6.424720
138241,1079.973439,13.0,-16.347993,0.356063,1.726132,-3.019206,-0.874072,0.369610,38.630940,10.208115,1.251905,-6.509489,-3.226304,9.915414,-0.952814,-0.255495,2.837007,58.352202,25.348942,5.154383,-9.862419,-5.322267,-4.816580,0.679191,-1.928497,-9.800714,3.794034,-25.074653,16.804234,-32.180863,7.087209,-2.269239,-0.486708,-1.732076,-2.223136,17.446077,7.989878,-6.761471
138242,1079.981251,13.0,-14.399766,0.916423,1.128389,-3.110302,-0.859553,0.248078,38.853007,10.105693,1.782533,-6.674558,-2.471423,6.993199,-0.803550,0.553607,2.883820,58.829264,24.523080,5.223109,-13.130319,-4.396824,-4.824985,0.623346,-1.756815,-10.142833,6.100014,-24.655264,16.739531,-30.130298,5.811476,-2.

Only include sensor data for model training and testing; remove time and activity code columns

In [58]:
df_2_data = df_2.iloc[:,2:] 
df_2_data

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
99845,-21.723717,-0.906709,-3.340406,-1.725156,1.383336,0.336675,37.023064,12.937910,-3.726508,-13.029014,2.780933,8.791527,-0.382982,-1.343675,-0.590641,59.231463,9.135267,10.539824,-7.421039,-11.276883,-7.374559,1.016987,-0.353633,1.914961,1.570668,-24.163764,12.631219,-5.127416,-3.676296,-3.275163,0.393977,-0.165850,-2.720616,23.882844,4.893803,-1.655445
99846,-26.138784,0.536051,2.104222,-2.643493,0.152196,0.345214,37.042200,13.303920,-3.233381,-19.080092,-4.158114,5.273474,-0.303717,0.264697,-0.726026,59.229017,9.047679,10.011694,-5.934692,-6.664719,-8.358840,1.667743,-0.268055,1.409116,0.899844,-24.119268,12.564553,-2.301847,-4.055829,-6.174088,0.606819,-0.320822,-2.267808,23.178881,5.153526,-1.746373
99847,-30.132945,3.394068,0.663351,-2.598406,0.614315,0.611814,36.813951,13.891790,-2.678930,-26.234258,-11.518528,2.172397,0.175134,0.943130,-0.927280,59.133957,9.197701,9.930569,-4.640564,-1.135332,-9.015020,2.075752,-0.265863,0.791366,0.318406,-24.269342,12.862920,-6.659033,-6.532508,-2.847930,0.738536,-0.389486,-1.764837,22.323508,5.576772,-1.843484
99848,-34.909282,2.884061,-1.588547,-2.502476,0.360988,0.486871,36.807108,14.106254,-1.983703,-29.278941,-1.742820,9.607354,1.012114,1.254744,-1.325094,58.996962,9.665787,10.196931,-3.737068,2.567238,-7.985579,2.310795,-0.278481,0.021561,0.128372,-24.128601,13.414327,-12.390128,-7.936329,-2.243952,0.631756,-0.313315,-1.445026,21.750232,5.577661,-1.967108
99849,-33.515009,2.409845,1.141210,-2.055668,0.487387,0.452026,37.177622,14.261348,-1.393292,-41.144383,-0.896207,10.917825,0.764393,3.136922,-0.738035,59.157526,9.984614,10.279470,-3.277699,3.577035,-6.633867,2.444098,-0.345753,-0.809418,-0.228798,-24.040231,13.630038,-11.429309,-6.475535,-4.453832,0.683026,-0.443690,-1.215093,21.108702,5.767338,-1.924253
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138240,-18.229356,0.024519,2.417356,-2.998924,-0.890793,0.371328,38.416287,10.314729,1.184247,-5.881049,-3.868557,11.655907,-1.176077,-1.188272,2.702903,58.044540,26.080097,5.584343,-5.484031,-5.405044,-5.335284,0.794465,-2.011347,-9.611934,1.082726,-25.105176,16.739606,-30.048477,4.258544,-2.326319,-0.821886,-1.550859,-2.450101,17.881782,7.909293,-6.424720
138241,-16.347993,0.356063,1.726132,-3.019206,-0.874072,0.369610,38.630940,10.208115,1.251905,-6.509489,-3.226304,9.915414,-0.952814,-0.255495,2.837007,58.352202,25.348942,5.154383,-9.862419,-5.322267,-4.816580,0.679191,-1.928497,-9.800714,3.794034,-25.074653,16.804234,-32.180863,7.087209,-2.269239,-0.486708,-1.732076,-2.223136,17.446077,7.989878,-6.761471
138242,-14.399766,0.916423,1.128389,-3.110302,-0.859553,0.248078,38.853007,10.105693,1.782533,-6.674558,-2.471423,6.993199,-0.803550,0.553607,2.883820,58.829264,24.523080,5.223109,-13.130319,-4.396824,-4.824985,0.623346,-1.756815,-10.142833,6.100014,-24.655264,16.739531,-30.130298,5.811476,-2.732883,0.178730,-1.444526,-1.877441,16.980908,7.835417,-6.881445
138243,-12.192350,0.878604,-0.139880,-3.385083,-1.056389,0.055782,38.809107,9.592058,1.687955,-6.45

In [59]:
# convert pandas dataframe to numpy array
load_X = df_2_data.to_numpy()
load_X.shape

(38400, 36)

## Set up sensors

In [60]:
from random import choice

lags = frequency # length of trajectory used to train LSTM; chose 128 for Ingraham data sampled at 128 Hz
n = load_X.shape[0] # total number of time steps (observations)
m = load_X.shape[1] # number of features per time step

time = np.arange(1, n+1, 1)

## Visualize IMU data

Observing raw data is important for understanding what is being used to train and test models. We visualize data using altair (alt). Two tutorials on some basic functionality are linked below:

* Long tutorial (1hr): https://youtu.be/umTwkgQoo_E

* Short tutorial (20min): https://youtu.be/o-nVM_FdIVc

Uncomment the line below when code is fully functioning to disable the 5000-row dataframe limit

In [61]:
# alt.data_transformers.disable_max_rows()

In [62]:
# set time to start at 0 (optional for clean viz)
time_zeroed = df_2.loc[:,"Time(s)__"] - df_2["Time(s)__"].iloc[0]

# view the first portion of the trial
df_2_data_reduced = df_2.head(1000)
df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)

# view the last portion of the trial
#df_2_data_reduced = df_2.tail(4000) 
#df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.tail(4000)

df_2_data_reduced.head()

/tmp/ipykernel_370674/3481633552.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)


,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,Time_Zeroed(s)
99845,780.016406,13.0,-21.723717,-0.906709,-3.340406,-1.725156,1.383336,0.336675,37.023064,12.937910,-3.726508,-13.029014,2.780933,8.791527,-0.382982,-1.343675,-0.590641,59.231463,9.135267,10.539824,-7.421039,-11.276883,-7.374559,1.016987,-0.353633,1.914961,1.570668,-24.163764,12.631219,-5.127416,-3.676296,-3.275163,0.393977,-0.165850,-2.720616,23.882844,4.893803,-1.655445,0.000000
99846,780.024218,13.0,-26.138784,0.536051,2.104222,-2.643493,0.152196,0.345214,37.042200,13.303920,-3.233381,-19.080092,-4.158114,5.273474,-0.303717,0.264697,-0.726026,59.229017,9.047679,10.011694,-5.934692,-6.664719,-8.358840,1.667743,-0.268055,1.409116,0.899844,-24.119268,12.564553,-2.301847,-4.055829,-6.174088,0.606819,-0.320822,-2.267808,23.178881,5.153526,-1.746373,0.007812
99847,780.032030,13.0,-30.132945,3.394068,0.663351,-2.598406,0.614315,0.611814,36.813951,13.891790,-2.678930,-26.234258,-11.518528,2.172397,0.175134,0.943130,-0.927280,59.133957,9.197701,9.930569,-4.640564,-1.135332,-9.015020,2.075752,-0.265863,0.791366,0.318406,-24.269342,12.862920,-6.659033,-6.532508,-2.847930,0.738536,-0.389486,-1.764837,22.323508,5.576772,-1.843484,0.015624
99848,780.039842,13.0,-34.909282,2.884061,-1.588547,-2.502476,0.360988,0.486871,36.807108,14.106254,-1.983703,-29.278941,-1.742820,9.607354,1.012114,1.254744,-1.325094,58.996962,9.665787,10.196931,-3.737068,2.567238,-7.985579,2.310795,-0.278481,0.021561,0.128372,-24.128601,13.414327,-12.390128,-7.936329,-2.243952,0.631756,-0.313315,-1.445026,21.750232,5.577661,-1.967108,0.023437
99849,780.047654,13.0,-33.515009,2.409845,1.141210,-2.055668,0.487387,0.452026,37.177622,14.261348,-1.393292,-41.144383,-0.896207,10.917825,0.764393,3.136922,-0.738035,59.157526,9.984614,10.279470,-3.277699,3.577035,-6.633867,2.444098,-0.345753,-0.809418,-0.228798,-24.040231,13.630038,-11.429309,-6.475535,-4.453832,0.683026,-0.443690,-1.215093,21.108702,5.767338,-1.924253,0.031249


Select which sensor location to visualize.

In [63]:
location = 'RightAnkle' # RightAnkle, LeftAnkle, Chest, Waist

In [64]:
# Define signal types and axes.
sensor = ['Acceleration', 'AngularVelocity', 'MagneticField']
dir = ['x','y','z']
plotStack = [0,0,0] # Preallocate plot for each signal

# Generate plots for each sensor type
for iSensor in range(len(sensor)): # loop through the signal types
    # create plots for x,y,z directions
    x_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_x', title = sensor[iSensor]),
        color = alt.value('#c6dbef')
    ).properties(
        width = 1000,
        height = 200
    )
    y_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_y', title = sensor[iSensor]),
        color = alt.value("#6baed6")
    )
    z_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_z', title = sensor[iSensor]),
        color = alt.value("#08519c")
    ).interactive()
    # Combine x,y,z plots
    plotStack[iSensor] = x_signal + y_signal + z_signal

alt.vconcat(plotStack[0], plotStack[1], plotStack[2]).properties(title = [location,""])

alt.VConcatChart(...)

# SHRED model function

In [65]:
### Generate input sequences to a SHRED model
def train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags):
  """
    Trains SHRED model for time series reconstruction

  Args: 
    transformed_X (numpy array): MinMax scaled dataset.
    sc (MinMaxScaler): Fitted MinMax scaler for inverse transformation.
    train_indices (array): Indices for the training set.
    valid_indices (array): Indices for the validation set.
    test_indices (array): Indices for the test set.
    sensor_locations (array): Column indices for sensor data.
    num_sensors (int): Number of signal measurements from sensors (e.g, triaxial = 3)
    m (int): Number of features per timestep
    n (int): Total number of time steps (observations)
    lags (int): length of trajectory
    
  Return:
    test_recons: Reconstructed data from the SHRED model on the test set
    test_ground_truth: Ground truth data from the test set
  """

  all_data_in = np.zeros((n - lags, lags, num_sensors))
  for i in range(len(all_data_in)):
      all_data_in[i] = transformed_X[i:i+lags, sensor_locations]
  ### Generate training validation and test datasets both for reconstruction of states and forecasting sensors
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
  valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
  test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

  ### -1 to have output be at the same time as final sensor measurements
  train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
  valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
  test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

  train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
  valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
  test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

  #shred = models.SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)

  sequence_length = frequency # 或者 test_dataset.X.shape[1]
  flattened_input_size = sequence_length * num_sensors
  shred = models.SDN(flattened_input_size, m, l1=350, l2=400, dropout=0.0).to(device)
  validation_errors = models.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=500, lr=1e-3, verbose=True, patience=5)

  # Generate reconstructions from the test set and print mean square error compared to the ground truth
  test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
  test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())

  return test_recons, test_ground_truth

# Train models

In [66]:
# partition into training, validation, test sets
train_indices, valid_indices, test_indices = ft.partition_data_seq(load_X, n, lags) 

# normalize input data using MinMaxScaler
transformed_X, sc = ft.transform_data(load_X, train_indices) 

### Define input sensor

In [67]:
# choose input sensor location
sensor_place = 'RightAnkle' # RightAnkle, Waist, or Chest

# choose input sensor type
sensor_path = '3acc_Training' # 3acc_Training, 3gyro_Training, 3acc3gyro_Training, or Xacc_Training

# access columns indices from main dataframe
sensor_locations, num_sensors = ft.sensor_loc_fun(sensor_path, sensor_place, df_2_data) # This function is specific to the dataset used in this project. Update it according the the types of signals (joint angles, EMG, etc) in your dataset.
train_names = [df_2_data.columns[i] for i in sensor_locations]

print('Number of signals: ', num_sensors)
print('Signals were chosen at: ', sensor_place)
print('Signals chosen: ', [df_2_data.columns[i] for i in sensor_locations])

Number of signals:  3
Signals were chosen at:  RightAnkle
Signals chosen:  ['RightAnkle_Acceleration_x', 'RightAnkle_Acceleration_y', 'RightAnkle_Acceleration_z']


In [68]:
# check path for saving dataframes
if trial_length == 6 and frequency == 128: # full-length trial, full frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ytest_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
else: # reduced trial length or frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'

print(os.path.isdir(save_test_df))
print(save_train_df)
print(save_test_df)

False
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC14/RightAnkle/3acc_Training/P03_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC14/RightAnkle/3acc_Training/P03_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv


In [69]:
test_indices

array([30657, 30658, 30659, ..., 38269, 38270, 38271], shape=(7615,))

In [70]:
# train SHRED model
Ypred, Ytest = train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags)

df_Ytest_SHRED = pd.DataFrame(Ytest, columns = df_2_data.columns)
df_Ypred_SHRED = pd.DataFrame(Ypred, columns = df_2_data.columns)

df_Ytest_SHRED['Type']='Measured'
df_Ypred_SHRED['Type']='SHRED'

df_Ytest_SHRED['Time']=df_2.iloc[test_indices + lags - 1,0].to_numpy()
df_Ypred_SHRED['Time']=df_2.iloc[test_indices + lags - 1,0].to_numpy()

# save dataframes as .csv if specified
#if save_df == True:
#  df_Ytest_SHRED.to_csv(save_train_df)
#  df_Ypred_SHRED.to_csv(save_test_df)


Training epoch 1
Error tensor(0.1518, device='cuda:0')
Training epoch 20
Error tensor(0.1221, device='cuda:0')
Training epoch 40
Error tensor(0.1144, device='cuda:0')
Training epoch 60
Error tensor(0.1141, device='cuda:0')
Training epoch 80
Error tensor(0.1010, device='cuda:0')
Training epoch 100
Error tensor(0.1025, device='cuda:0')
Training epoch 120
Error tensor(0.1023, device='cuda:0')
Training epoch 140
Error tensor(0.0994, device='cuda:0')
Training epoch 160
Error tensor(0.0995, device='cuda:0')
Training epoch 180
Error tensor(0.1018, device='cuda:0')
Training epoch 200
Error tensor(0.1001, device='cuda:0')
Training epoch 220
Error tensor(0.1019, device='cuda:0')
Training epoch 240
Error tensor(0.1000, device='cuda:0')


# Visualize Results

In [75]:
df_SHRED_tidy = ft.concatRaw(1,sensor_place, sensor_path, 1, 1, df_Ypred_SHRED, df_Ytest_SHRED ,subj)

In [76]:
print("df_SHRED_tidy 的列名:", df_SHRED_tidy.columns)
print("df_SHRED_tidy 的头部数据:\n", df_SHRED_tidy.tail(100))

df_SHRED_tidy 的列名: Index(['Subject', 'Pred', 'True', 'Input Location', 'Sensor Type',
       'Output Location', 'Output Signal', 'Output Direction', 'Output Axis',
       'Assessment', 'Time'],
      dtype='object')
df_SHRED_tidy 的头部数据:
      Subject       Pred       True Input Location             Sensor Type  \
7100      03   5.316804   4.986156     RightAnkle  Triaxial Accelerometer   
7101      03 -15.633347 -17.565411     RightAnkle  Triaxial Accelerometer   
7102      03   6.461894   8.009828     RightAnkle  Triaxial Accelerometer   
7103      03   8.919059   7.708647     RightAnkle  Triaxial Accelerometer   
7104      03   1.919397   1.493967     RightAnkle  Triaxial Accelerometer   
...      ...        ...        ...            ...                     ...   
7195      03  -0.169415   0.068398     RightAnkle  Triaxial Accelerometer   
7196      03   4.194623   4.043372     RightAnkle  Triaxial Accelerometer   
7197      03  -0.772322  -1.260704     RightAnkle  Triaxial Accelerom

In [77]:
# format: ft.extractSignal(output_location, output_signal, output_axis, df_SHRED_tidy)
    # output_location: 'Chest', 'Waist', 'RightAnkle', 'LeftAnkle'
    # output_signal: 'Acceleration', 'Angular Velocity', 'Magnetic Field'
    # output_axis: 'x', 'y', z'

Signal1 = ft.extractSignal('LeftAnkle', 'Acceleration', 'x', df_SHRED_tidy)
Signal2 = ft.extractSignal('Chest', 'Acceleration', 'x', df_SHRED_tidy)
Signal3 = ft.extractSignal('Waist', 'Acceleration', 'x', df_SHRED_tidy)

my_scheme = ['#1e88e5', "#6E6E6E"] # '#014337', '#1e88e5', '#DB1048'

# Compute error: ft.rmse_error, ft.mae_error, OR ft.mbe_error
Signal1_error = ft.rmse_error(Signal1[Signal1['Type'] == 'True']['Value'], Signal1[Signal1['Type'] == 'SHRED']['Value'])
Signal2_error = ft.rmse_error(Signal2[Signal2['Type'] == 'True']['Value'], Signal2[Signal2['Type'] == 'SHRED']['Value'])
Signal3_error = ft.rmse_error(Signal3[Signal3['Type'] == 'True']['Value'], Signal3[Signal3['Type'] == 'SHRED']['Value'])

# plot left ankle acceleration
line1 = alt.Chart(Signal1).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 1 LightAnkle: RMSE = {Signal1_error:.2f}'  # Can change this title to be specific to the output signal
)
# plot chest acceleration
line2 = alt.Chart(Signal2).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 2: Chest RMSE = {Signal2_error:.2f}'  # Can change this title to be specific to the output signal
)

# plot Waist acceleration
line3 = alt.Chart(Signal3).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 3: Waist RMSE = {Signal3_error:.2f}'  # Can change this title to be specific to the output signal
)

final_chart = alt.vconcat(line1, line2, line3).properties(
    title=f'Parameter = {save_tag}, Right Ankle Input' # Can change this title to match the input sensor
    # increase font size
).configure_axis(
    labelFontSize=18,
    titleFontSize=20
).configure_title(
    fontSize=24
).configure_legend(
    labelFontSize=18,
    titleFontSize=20
)

final_chart

alt.VConcatChart(...)